In [ ]:
# 逆行列：基本変形の行列
import numpy as np

def p(n,a,b):
    """ 左から掛けると a 行目と b 行目を入れ替える行列 """
    e = np.identity(n)
    order = list(range(n))
    order[a] = b
    order[b] = a
    return e[order,:]

def q(n,a,c):
    """ 左から掛けると a 行目を c 倍する行列 """
    e = np.identity(n)
    e[a,a] = c
    return e

def r(n,a,b,c):
    """ 左から掛けると a 行目を c 倍したものを b 行目に加える行列 """
    e = np.identity(n)
    e[b,a] = c
    return e

In [ ]:
# 逆行列：掃き出し法
def inv(x):
    n,_ = x.shape
    e = np.identity(n)
    for i in range(n):
        # i 列目の i 行目以下で，絶対値が最大の成分を持つ行を選ぶ（ピボット選択）
        u = i + np.argmax(np.abs(x[i:,i]))
        if abs(x[u,i]) < 1e-12:      # その列が（ほぼ）0 なら，正則でない
            return None
        v = p(n,i,u);         x = v@x; e = v@e
        v = q(n,i,1/x[i,i]);  x = v@x; e = v@e
        for j in range(n):
            if i != j:
                v = r(n,i,j,-x[j,i]); x = v@x; e = v@e
    return e

x = np.random.random([9,9])
np.set_printoptions(precision=3, suppress=True)
y = inv(x)
print(np.allclose(x@y, np.identity(9)), np.allclose(y, np.linalg.inv(x)))

In [ ]:
# テスト：ランダムな行列以外でも確かめる
tests = [np.array([[0., 1.], [1., 0.]]),        # 左上が 0 の正則行列（行の入れ替えが必要）
         0.0001 * np.identity(2),              # 成分は小さいが正則
         np.array([[1., 2.], [2., 4.]]),        # 正則でない
         np.array([[2., 1., 1.], [4., 2., 3.], [1., 5., 7.]])]
for t in tests:
    y = inv(t)
    if y is None:
        print("正則でない:", np.linalg.det(t))
    else:
        print(np.allclose(t@y, np.identity(len(t))))

**注意** 以前の解答では，ピボットの選択に `np.argmax(map(abs, x[i:,i]))` を使っていましたが，np.argmax に map を渡すと，配列として扱われずに常に 0 が返ります。そのため行の入れ替えが行われず，`[[0, 1], [1, 0]]` のような正則行列でも None を返していました。また，「0.001 より小さければ正則でない」という判定では，成分の小さな正則行列も正則でないと判定されてしまいます。

In [ ]:
# 練習問題（行列そのものを書き換える版）
# x の右に単位行列をつけた n x 2n 行列 z を作り，z の行そのものを書き換える。
def inv2(x):
    n,_ = x.shape
    z = np.zeros((n, 2*n))
    z[:, :n] = x
    z[:, n:] = np.identity(n)
    for i in range(n):
        u = i + np.argmax(np.abs(z[i:,i]))
        if abs(z[u,i]) < 1e-12:
            return None
        z[[i, u], :] = z[[u, i], :]      # 行の入れ替え（p に対応）
        z[i, :] = z[i, :] / z[i, i]      # i 行目を割る（q に対応）
        for j in range(n):
            if i != j:
                z[j, :] = z[j, :] - z[j, i] * z[i, :]   # r に対応
    return z[:, n:]

x = np.random.random([9,9])
print(np.allclose(inv2(x), inv(x)))
for t in tests:
    y = inv2(t)
    print(None if y is None else np.allclose(t@y, np.identity(len(t))))